In [ ]:
!pip install langchain langchain-community langchain-groq chromadb \ sentence-transformers pypdf -q

In [ ]:
!pip install -q \
langchain \
langchain-community \
langchain-core \
langchain-text-splitters \
langchain-groq \
langchain-huggingface \
chromadb \
sentence-transformers \
pypdf

In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print("Uploaded:", pdf_path)

Saving System Design Proposal.pdf to System Design Proposal.pdf
Uploaded: System Design Proposal.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Total Pages:", len(documents))

Total Pages: 3


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

texts = text_splitter.split_documents(documents)

print("Total Chunks:", len(texts))

Total Chunks: 21


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory="chroma_db"
)

print("Vector Database Created Successfully!")

Vector Database Created Successfully!


In [ ]:
def answer_from_pdf(query):

    docs = db.as_retriever().invoke(query)

    context = "\n".join(doc.page_content for doc in docs)

    prompt = f"""
Use ONLY this context to answer.

{context}

Question:
{query}

If the context does not contain the answer, reply exactly:

I don't know
"""

    response = llm.invoke(prompt)

    return response.content

In [ ]:
def adaptive_answer(question, max_tries=3):

    query = question

    for attempt in range(1, max_tries + 1):

        print(f"Attempt {attempt} with query: {query}")

        answer = answer_from_pdf(query)

        if "i don't know" not in answer.lower():
            return answer

        query = llm.invoke(
            f"Rephrase this search query differently: {query}"
        ).content

    return "Could not find an answer after several tries."

In [ ]:
print(adaptive_answer("What is the main conclusion of the document?"))

Attempt 1 with query: What is the main conclusion of the document?
Attempt 2 with query: Here are a few alternative search queries:

1. What is the key takeaway from the document?
2. What is the author's main point or finding?
3. What is the summary or conclusion of the document?
4. What is the central argument or thesis of the document?
5. What is the final verdict or recommendation of the document?

These rephrased search queries can help you refine your search and get more accurate results.
1. What is the key takeaway from the document?
The key takeaway is that Orion Intelligence will use a hierarchical Supervisor–Worker Architecture to scale out specialized team operations.

2. What is the author's main point or finding?
The author's main point is to describe how Orion Intelligence will use a hierarchical Supervisor–Worker Architecture to scale out specialized team operations.

3. What is the summary or conclusion of the document?
The summary is that Orion Intelligence will use a h